In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, utils
from tensorflow.keras.datasets import cifar10

# --- 1. 数据准备 (CIFAR-10) ---
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# 归一化 (让像素值在 0-1 之间)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 独热编码 (One-Hot Encoding)，用于多分类
y_train = utils.to_categorical(y_train, 10)
y_test = utils.to_categorical(y_test, 10)

# 获取输入形状 (CIFAR-10 是 32x32 的 3 通道彩色图像)
input_shape = x_train.shape[1:]

# --- 2. 搭建 Inception 块 (实验核心) ---
def inception_block(x_in, filters):
    """
    一个简化的 Inception v1 块
    """
    # Inception 块有4个并行的分支

    # 分支 1: 1x1 卷积
    # 1x1 卷积主要用于“降维”和跨通道信息融合
    branch_1x1 = layers.Conv2D(filters[0], (1, 1), padding='same', activation='relu')(x_in)

    # 分支 2: 1x1 卷积 -> 3x3 卷积
    # 先用 1x1 降维，再用 3x3 提取特征，这是经典操作
    branch_3x3 = layers.Conv2D(filters[1], (1, 1), padding='same', activation='relu')(x_in)
    branch_3x3 = layers.Conv2D(filters[2], (3, 3), padding='same', activation='relu')(branch_3x3)

    # 分支 3: 1x1 卷积 -> 5x5 卷积
    # (在现代网络中，5x5 常用两个 3x3 代替，这里按原始 Inception 简写)
    branch_5x5 = layers.Conv2D(filters[3], (1, 1), padding='same', activation='relu')(x_in)
    branch_5x5 = layers.Conv2D(filters[4], (5, 5), padding='same', activation='relu')(branch_5x5)

    # 分支 4: 3x3 最大池化 -> 1x1 卷积
    branch_pool = layers.MaxPooling2D((3, 3), strides=(1, 1), padding='same')(x_in)
    branch_pool = layers.Conv2D(filters[5], (1, 1), padding='same', activation='relu')(branch_pool)

    # 沿通道维度 (axis=3) 拼接所有分支的输出
    x_out = layers.concatenate([branch_1x1, branch_3x3, branch_5x5, branch_pool], axis=3)

    return x_out

# --- 3. 搭建完整模型 (使用 Inception 块) ---
def build_googlenet_mini(input_shape):
    inputs = layers.Input(shape=input_shape)

    # 1. 一个简单的“主干” (Stem)
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)

    # 2. 插入我们要求的 Inception 块！
    # [64, 96, 128, 16, 32, 32] 是给 Inception 块 4 个分支的 6 个滤波器数量
    x = inception_block(x, filters=[64, 96, 128, 16, 32, 32])

    # 3. 分类器
    x = layers.GlobalAveragePooling2D()(x) # 全局平均池化，代替 Flatten
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(10, activation='softmax')(x) # 10个类别

    model = models.Model(inputs=inputs, outputs=outputs, name="GoogLeNet_Mini")
    return model

# --- 4. 编译和训练 ---
model_googlenet = build_googlenet_mini(input_shape)
model_googlenet.summary() # 打印模型结构

# 编译模型
model_googlenet.compile(
    optimizer='adam',
    loss='categorical_crossentropy', # 多分类交叉熵
    metrics=['accuracy']
)

print("\n--- 开始训练 GoogLeNet (Inception) 模型 ---")
# 训练模型，根据实验要求设置 epochs=5
model_googlenet.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(x_test, y_test)
)

# 评估模型
print("\n--- 评估模型 ---")
loss, acc = model_googlenet.evaluate(x_test, y_test)
print(f"测试集准确率 (Test Accuracy): {acc:.4f}")

Model: "GoogLeNet_Mini"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 16, 16,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 16, 16,    │      6,240 │ max_pooling2d[0]… │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │      1,040 │ max_pooling2d[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 16, 16,    │          0 │ max_pooling2d[0]… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 16, 16,    │      4,160 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 16, 16,    │    110,720 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 16, 16,    │     12,832 │ conv2d_4[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 16, 16,    │      2,080 │ max_pooling2d_1[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16, 16,    │          0 │ conv2d_1[0][0],   │
│ (Concatenate)       │ 256)              │            │ conv2d_3[0][0],   │
│                     │                   │            │ conv2d_5[0][0],   │
│                     │                   │            │ conv2d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ concatenate[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     32,896 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 10)        │      1,290 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 173,050 (675.98 KB)

 Trainable params: 173,050 (675.98 KB)

 Non-trainable params: 0 (0.00 B)


--- 开始训练 GoogLeNet (Inception) 模型 ---
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 21ms/step - accuracy: 0.2924 - loss: 1.8580 - val_accuracy: 0.3872 - val_loss: 1.6380
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.4030 - loss: 1.6056 - val_accuracy: 0.4353 - val_loss: 1.5424
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 16s 21ms/step - accuracy: 0.4593 - loss: 1.4706 - val_accuracy: 0.4767 - val_loss: 1.4229
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - accuracy: 0.5003 - loss: 1.3681 - val_accuracy: 0.5019 - val_loss: 1.3550
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - accuracy: 0.5303 - loss: 1.2894 - val_accuracy: 0.5316 - val_loss: 1.2870

--- 评估模型 ---
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5316 - loss: 1.2870
测试集准确率 (Test Accuracy): 0.5316


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, utils
from tensorflow.keras.datasets import cifar10

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# 归一化
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = utils.to_categorical(y_train, 10)
y_test = utils.to_categorical(y_test, 10)

input_shape = x_train.shape[1:]


def inception_block(x_in, filters):
    """
    简化Inception v1 块
    """

    # 分支 1: 1x1 卷积
    # 用于“降维”和跨通道信息融合
    branch_1x1 = layers.Conv2D(filters[0], (1, 1), padding="same", activation="relu")(
        x_in
    )

    # 分支 2: 1x1 卷积 -> 3x3 卷积
    branch_3x3 = layers.Conv2D(filters[1], (1, 1), padding="same", activation="relu")(
        x_in
    )
    branch_3x3 = layers.Conv2D(filters[2], (3, 3), padding="same", activation="relu")(
        branch_3x3
    )

    # 分支 3: 1x1 卷积 -> 5x5 卷积
    branch_5x5 = layers.Conv2D(filters[3], (1, 1), padding="same", activation="relu")(
        x_in
    )
    branch_5x5 = layers.Conv2D(filters[4], (5, 5), padding="same", activation="relu")(
        branch_5x5
    )

    # 分支 4: 3x3 最大池化 -> 1x1 卷积
    branch_pool = layers.MaxPooling2D((3, 3), strides=(1, 1), padding="same")(x_in)
    branch_pool = layers.Conv2D(filters[5], (1, 1), padding="same", activation="relu")(
        branch_pool
    )

    # 沿通道维度 (axis=3) 拼接所有分支的输出
    x_out = layers.concatenate(
        [branch_1x1, branch_3x3, branch_5x5, branch_pool], axis=3
    )

    return x_out


def build_googlenet_mini(input_shape):
    inputs = layers.Input(shape=input_shape)

    # 1. 主干
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D((2, 2))(x)

    # [64, 96, 128, 16, 32, 32] 是给 Inception 块 4 个分支的 6 个滤波器数量
    x = inception_block(x, filters=[64, 96, 128, 16, 32, 32])

    # 3. 分类器
    x = layers.GlobalAveragePooling2D()(x)  # 全局平均池化
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(10, activation="softmax")(x)  # 10个类别

    model = models.Model(inputs=inputs, outputs=outputs, name="GoogLeNet_Mini")
    return model


model_googlenet = build_googlenet_mini(input_shape)
model_googlenet.summary()

model_googlenet.compile(
    optimizer="adam",
    loss="categorical_crossentropy",  # 多分类交叉熵
    metrics=["accuracy"],
)

print("\n--- 开始训练 GoogLeNet (Inception) 模型 ---")
# 训练模型，epochs=5
model_googlenet.fit(
    x_train, y_train, epochs=5, batch_size=64, validation_data=(x_test, y_test)
)

print("\n--- 评估模型 ---")
loss, acc = model_googlenet.evaluate(x_test, y_test)
print(f"测试集准确率 (Test Accuracy): {acc:.4f}")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, utils
from tensorflow.keras.datasets import cifar10

# --- 1. 数据准备 (CIFAR-10) ---
# (假设已在上一实验加载和预处理，若单独运行，请取消注释)
# (x_train, y_train), (x_test, y_test) = cifar10.load_data()
# x_train = x_train.astype('float32') / 255.0
# x_test = x_test.astype('float32') / 255.0
# y_train = utils.to_categorical(y_train, 10)
# y_test = utils.to_categorical(y_test, 10)
# input_shape = x_train.shape[1:]

# --- 2. 搭建 ResNet 块 (实验核心) ---
def resnet_block(x_in, filters, kernel_size=(3, 3), strides=(1, 1)):
    """
    一个简化的 ResNet 块 (Identity Block)
    """
    # "捷径" (shortcut) 先保存好输入
    shortcut = x_in

    # ResNet 的主路径 (F(x))
    x = layers.Conv2D(filters, kernel_size, strides=strides, padding='same')(x_in)
    x = layers.BatchNormalization()(x) # BN 是 ResNet 的标配
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)

    # 关键一步：把主路径的输出 F(x) 和 "捷径" x 加起来
    # 注意：这里我们假设 x_in 和 x 的维度相同，所以可以直接相加
    # (如果维度不同，shortcut 需要一个 1x1 卷积来变换维度，那叫 'Convolution Block')
    x = layers.add([x, shortcut])

    # 最后的激活函数
    x_out = layers.Activation('relu')(x)

    return x_out

# --- 3. 搭建完整模型 (使用 ResNet 块) ---
def build_resnet_mini(input_shape):
    inputs = layers.Input(shape=input_shape)

    # 1. 一个简单的“主干” (Stem)
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)

    # 2. 插入我们要求的 ResNet 块！
    # 我们使用 64 个滤波器
    x = resnet_block(x, filters=64)
    # 还可以再多叠几层
    x = resnet_block(x, filters=64)

    # 3. 分类器
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(10, activation='softmax')(x) # 10个类别

    model = models.Model(inputs=inputs, outputs=outputs, name="ResNet_Mini")
    return model

# --- 4. 编译和训练 ---
model_resnet = build_resnet_mini(input_shape)
model_resnet.summary() # 打印模型结构

# 编译模型
model_resnet.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- 开始训练 ResNet (Residual) 模型 ---")
# 训练模型，根据实验要求设置 epochs=5
model_resnet.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(x_test, y_test)
)

# 评估模型
print("\n--- 评估模型 ---")
loss, acc = model_resnet.evaluate(x_test, y_test)
print(f"测试集准确率 (Test Accuracy): {acc:.4f}")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, utils
from tensorflow.keras.datasets import cifar10

def resnet_block(x_in, filters, kernel_size=(3, 3), strides=(1, 1)):
    """
    简化ResNet
    """
    shortcut = x_in

    # ResNet主路径
    x = layers.Conv2D(filters, kernel_size, strides=strides, padding="same")(x_in)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, kernel_size, padding="same")(x)
    x = layers.BatchNormalization()(x)

    x = layers.add([x, shortcut])

    x_out = layers.Activation("relu")(x)

    return x_out


def build_resnet_mini(input_shape):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D((2, 2))(x)

    # 64 个滤波器
    x = resnet_block(x, filters=64)
    # 多叠几层
    x = resnet_block(x, filters=64)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(10, activation="softmax")(x)  # 10个类别

    model = models.Model(inputs=inputs, outputs=outputs, name="ResNet_Mini")
    return model


model_resnet = build_resnet_mini(input_shape)
model_resnet.summary()

model_resnet.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

print("\n--- 开始训练 ResNet (Residual) 模型 ---")
# 训练模型epochs=5
model_resnet.fit(
    x_train, y_train, epochs=5, batch_size=64, validation_data=(x_test, y_test)
)

print("\n--- 评估模型 ---")
loss, acc = model_resnet.evaluate(x_test, y_test)
print(f"测试集准确率 (Test Accuracy): {acc:.4f}")

Model: "ResNet_Mini"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 32, 32,    │      1,792 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 16, 16,    │     36,928 │ max_pooling2d_2[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 16, 16,    │        256 │ conv2d_8[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 16, 16,    │     36,928 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        256 │ conv2d_9[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 16, 16,    │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 16, 16,    │     36,928 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        256 │ conv2d_10[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 16, 16,    │     36,928 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        256 │ conv2d_11[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 16, 16,    │          0 │ add_1[0][0]     

 Total params: 160,138 (625.54 KB)

 Trainable params: 159,626 (623.54 KB)

 Non-trainable params: 512 (2.00 KB)


--- 开始训练 ResNet (Residual) 模型 ---
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 24s 28ms/step - accuracy: 0.5000 - loss: 1.3547 - val_accuracy: 0.5022 - val_loss: 1.3779
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.6405 - loss: 1.0014 - val_accuracy: 0.2855 - val_loss: 2.3584
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 24s 31ms/step - accuracy: 0.6868 - loss: 0.8747 - val_accuracy: 0.5053 - val_loss: 1.4425
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 24s 31ms/step - accuracy: 0.7267 - loss: 0.7691 - val_accuracy: 0.6809 - val_loss: 0.8987
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 24s 30ms/step - accuracy: 0.7554 - loss: 0.6959 - val_accuracy: 0.6494 - val_loss: 1.0729

--- 评估模型 ---
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6494 - loss: 1.0729
测试集准确率 (Test Accuracy): 0.6494
